In [ ]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

from pathlib import Path
import torch
from ultralytics import YOLO

In [ ]:
print(f"CUDA Kullanılabilir mi: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Aktif GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
ROOT_DIR = Path.cwd().parent
yaml_path = ROOT_DIR / "configs/steel_yolo.yaml"

In [ ]:
model = YOLO("yolov8n.pt")

In [ ]:
yaml_path = ROOT_DIR / "configs/steel_yolo.yaml"

# Modeli yükle ve 25 epoch eğit
model = YOLO("yolov8n.pt")

results = model.train(
    data=str(yaml_path.resolve()),
    epochs=25,
    imgsz=224,
    batch=32,
    device=0,           # RTX 3050 GPU
    workers=2,
    project=str(ROOT_DIR / "models/yolo_runs"),
    name="baseline_yolov8n",
    exist_ok=True,
    plots=True
)

In [ ]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

from pathlib import Path
import cv2
import matplotlib.pyplot as plt
import json
from ultralytics import YOLO

ROOT_DIR = Path.cwd().parent

model_path = ROOT_DIR / "models/yolo_runs/baseline_yolov8n/weights/best.pt"
model = YOLO(str(model_path))

test_img_dir = ROOT_DIR / "data/processed/yolo/images/val"
sample_images = list(test_img_dir.glob("scratches_*.jpg"))
if not sample_images:
    sample_images = list(test_img_dir.glob("*.jpg"))

test_image_path = sample_images[0]
print(f"Test Edilen Görsel: {test_image_path.name}")

results = model.predict(
    source=str(test_image_path),
    conf=0.30,
    iou=0.50,
    device=0,
    verbose=False
)

result = results[0]

inspection_payload = {
    "image_name": test_image_path.name,
    "image_width": result.orig_shape[1],
    "image_height": result.orig_shape[0],
    "total_defects_found": len(result.boxes),
    "defects": []
}

for box in result.boxes:
    xyxy = box.xyxy[0].cpu().numpy().tolist()
    conf = float(box.conf[0].cpu().numpy())
    cls_id = int(box.cls[0].cpu().numpy())
    cls_name = model.names[cls_id]
    
    inspection_payload["defects"].append({
        "class_id": cls_id,
        "class_name": cls_name,
        "confidence": round(conf, 4),
        "bbox": [round(coord, 2) for coord in xyxy]
    })

print("\n--- Java / Monitoring Servisine İletilecek Telemetri Yükü ---")
print(json.dumps(inspection_payload, indent=2))

res_plotted = result.plot() # Ultralytics'in etiketleri ve kutuları çizdiği BGR matrisi
res_rgb = cv2.cvtColor(res_plotted, cv2.COLOR_BGR2RGB)

plt.figure(figsize=(7, 7))
plt.imshow(res_rgb)
plt.axis("off")
plt.title(f"Tespit: {test_image_path.name} ({len(result.boxes)} Kusur Bulundu)", fontweight='bold')
plt.show()

In [ ]:

metrics = model.val()

print("\n--- Sınıf Bazlı mAP50 Skorları ---")
for idx, name in model.names.items():
    ap50 = metrics.box.ap50[idx]
    print(f"{name:<18} : %{ap50 * 100:.2f}")

print(f"\nGenel mAP50        : %{metrics.box.map50 * 100:.2f}")
print(f"Genel mAP50-95     : %{metrics.box.map * 100:.2f}")